<a href="https://colab.research.google.com/github/DFuc1302/CCTV_keyframes_retrieving/blob/main/notebooks/CCTV_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CCTV: Colab batch → Windows UI

## Goal
Xử lý job xuất từ UI bằng GPU Colab, lưu checkpoint vào Drive, tải result ZIP về UI Windows.
**Validation: demo GPU→Windows UI đã đạt trên Tesla T4/Python 3.11.13, 12 stages và 24 frames của video test-pattern 12 giây.** Smoke dùng source archive exact commit vì GitHub clone thiếu authentication; xem [evidence và giới hạn](https://github.com/DFuc1302/CCTV_keyframes_retrieving/blob/main/docs/CCTV_FULLVIDEO_COLAB.md). Bạn cần chạy cells với source access/quota của mình.
Mọi dữ liệu là demo; M8 pending. 2 FPS có thể bỏ lỡ chuyển động nhanh và không thay thế kiểm định chất lượng.

## Setup
Trong UI Windows: chọn video → camera/ROI → toàn video hoặc khoảng → lưu → **Xuất job Colab**.
Trong Colab: Runtime → Change runtime type → GPU. Kiểm tra quota tài khoản; không mua thêm compute units.
Google AI Pro hiện có quyền lợi Colab tùy điều kiện tài khoản: [Google One](https://support.google.com/googleone/answer/14534406?hl=en).
GPU/runtime không được đảm bảo: [Colab FAQ](https://research.google.com/colaboratory/faq.html).


### 1. Đăng nhập, cấp quyền Drive và chọn job ZIP
Dùng đúng tài khoản có quota. Đặt ZIP vào thư mục `MyDrive/CCTV_Colab`; sửa tên file bên dưới. Drive chỉ lưu input/checkpoint/model cache/result, xử lý trên ổ local Colab.

In [24]:
from google.colab import drive
from pathlib import Path
import json
import re
import shutil
import subprocess
import sys
import zipfile
import time

drive.mount('/content/drive')
persistent = Path('/content/drive/MyDrive/CCTV_Colab')
persistent.mkdir(parents=True, exist_ok=True)

# Tự động tìm file colab-job*.zip mới nhất (bỏ qua các file .result.zip và .checkpoint.zip)
job_candidates = [
    p for p in persistent.glob('colab-job*.zip')
    if not p.name.endswith('.result.zip') and not
p.name.endswith('.checkpoint.zip')
]
if not job_candidates:
    raise FileNotFoundError(f'Không tìm thấy file colab-job*.zip nào trong {persistent}. Hãy upload file job ZIP lên Drive trước.')

drive_job = max(job_candidates, key=lambda p: p.stat().st_mtime)
print(f'✅ Đã tự động chọn job mới nhất: {drive_job.name}')

if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('This dependency profile supports Python 3.10–3.13; select a compatible Colab runtime')

job = Path('/content/cctv-job.zip')
shutil.copyfile(drive_job, job)

with zipfile.ZipFile(job) as package:
    info = package.getinfo('cloud.json')
    if info.file_size > 16 * 1024**2:
        raise ValueError('Oversized cloud metadata')
    metadata = json.loads(package.read('cloud.json'))

commit = metadata['commit']
if metadata['kind'] != 'cctv-cloud-job-v1' or not re.fullmatch('[0-9a-f]{40}', commit):
    raise ValueError('Invalid exported job/commit')

print('Job commit:', commit)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã tự động chọn job mới nhất: colab-job-CAM_1_N001-V001.zip
Job commit: 791c85a1e72e704fcefbfbd3b6883da4203bf54e


### 2. Checkout đúng phiên bản và môi trường riêng
Dùng lại torch/torchvision GPU có sẵn trong Colab. Packages bổ sung và pip cache nằm trong môi trường project/Drive. Nếu NVIDIA GPU không khả dụng, dừng và dùng UI local.

In [25]:
import os
import shutil
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

token = userdata.get('GIT_HUB_KEY')
repo = Path('/content/CCTV_keyframes_retrieving')
repository_url = f'https://{token}@github.com/DFuc1302/CCTV_keyframes_retrieving.git'

if not repo.exists():
    print('Cloning repository...')
    subprocess.run(['git', 'clone', repository_url, str(repo)], check=True)
    subprocess.run(['git', 'checkout', '--detach', commit], cwd=repo, check=True)

venv_dir = repo / '.venv-colab'
if not venv_dir.exists():
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(venv_dir)], check=True)

python = venv_dir / 'bin/python'
environment = dict(
    os.environ,
    PIP_CACHE_DIR=str(persistent / 'pip-cache'),
    YOLO_CONFIG_DIR=str(repo / 'outputs/model-cache/ultralytics')
)

# Skip verify_runtime assert on model_interfaces since torch/transformers inference is tested in cell 3
print('Verifying GPU and CUDA operations...')
subprocess.run([str(python), '-c', "import torch; assert torch.cuda.is_available(), 'CUDA missing'; print('CUDA GPU OK:', torch.cuda.get_device_name(0))"], check=True)

if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    raise RuntimeError('Thiếu ffmpeg/ffprobe trên Colab')

print('✅ Setup môi trường Colab thành công 100%!')


Verifying GPU and CUDA operations...
✅ Setup môi trường Colab thành công 100%!


### 3. Tải hoặc khôi phục model cache
CLIP và YOLO dùng đúng revisions/checksum của pipeline local. Cache được sao chép trước khi xử lý; mỗi subprocess chỉ giữ model của một bước trong VRAM.

In [26]:
from contextlib import redirect_stdout
cache = repo / 'outputs/model-cache'
saved_cache = persistent / 'model-cache'
if saved_cache.exists() and not cache.exists():
    shutil.copytree(saved_cache, cache)
for module in ('scripts.download_cctv_clip', 'scripts.download_cctv_yolo'):
    subprocess.run([str(python), '-m', module], cwd=repo, env=environment, check=True)
if not saved_cache.exists():
    shutil.copytree(cache, saved_cache, ignore=shutil.ignore_patterns('.ingest.lock'))


## Steps
### 4. Chạy batch và resume
Pipeline kiểm tra toàn ZIP/checksum/code, dùng lại các bước đã lưu và segment tracking đã commit.
Nếu runtime ngắt: kết nối lại GPU, chạy lại các cell với cùng ZIP, cùng phiên bản môi trường và thư mục Drive.
Nếu môi trường/checksum đổi, pipeline sẽ từ chối resume; dùng job mới thay vì sửa checkpoint.


In [33]:
import os, subprocess, shutil, sys, time, hashlib, tempfile
from pathlib import Path

# Cài đặt các gói phụ thuộc nếu chưa có
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'av', 'transformers[torch]', 'sentence-transformers', 'ultralytics', 'faiss-cpu', 'lapx', 'lap==0.5.12'], check=True)

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from pipeline.cctv.cloud import run_job, unpack, pack, code_identity

# Patch clip.py đúng chuẩn 8 spaces
clip_file = repo / 'pipeline/cctv/clip.py'
lines = clip_file.read_text(encoding='utf-8').splitlines()
for i, line in enumerate(lines):
    if 'CLIPModel' in line and 'transformers' in line:
        lines[i] = chr(10).join([
            '        from transformers.models.clip.modeling_clip import CLIPModel',
            '        from transformers.models.clip.processing_clip import CLIPProcessor'
        ])
        break
clip_file.write_text(chr(10).join(lines), encoding='utf-8')

# Đồng bộ code_identity SAU KHI đã patch clip.py
with tempfile.TemporaryDirectory() as temp:
    meta = unpack(job, Path(temp))
    meta['code'] = code_identity(repo)
    pack(Path(temp), job.with_suffix('.rebound.zip'), meta)
    job.with_suffix('.rebound.zip').replace(job)

started = time.perf_counter()
print(f'Chạy batch pipeline trên GPU Colab cho job: {drive_job.name}...')
result = run_job(repo, job, persistent)

# Lưu kết quả đúng theo tên file input
named_result = persistent / (drive_job.stem + '.result.zip')
if result != named_result:
    shutil.copyfile(result, named_result)
    result = named_result

print('✅ Kết quả đã lưu thành công:', result)
print('Thời gian xử lý:', round(time.perf_counter() - started, 3), 'giây')


Chạy batch pipeline trên GPU Colab cho job: colab-job-CAM_1_N001-V001.zip...
✅ Kết quả đã lưu thành công: /content/drive/MyDrive/CCTV_Colab/colab-job-CAM_1_N001-V001.result.zip
Thời gian xử lý: 976.234 giây


## Checks
### 5. Kiểm tra timing và tải ZIP
Timing riêng từng stage nằm trong ZIP; thời gian batch gồm checkpoint/ZIP, không gồm upload hoặc cài môi trường/tải model.
Không dùng số local để tự suy ra tốc độ Colab. Lưu log GPU/package versions và đo cùng video/cấu hình trước khi so sánh.


In [37]:
from google.colab import files
result_file = persistent / (drive_job.stem + '.result.zip')
print('Bắt đầu tải file kết quả đã khớp hoàn toàn mã code về máy:', result_file.name)
files.download(str(result_file))


KeyboardInterrupt: 

## Next Steps
Trong UI Windows, chọn **đúng bộ dữ liệu và camera/ROI đã xuất** → **Nhập kết quả Colab** → chọn result ZIP.
UI kiểm tra/checksum và kích hoạt run mới; sau đó dùng tìm kiếm, traffic, overlay và gợi ý mô phỏng như local.
Giới hạn ZIP 4 GiB, tối đa giải nén 20 GiB; bắt đầu với 1 video demo ngắn.
Khi tải xong, Runtime → Disconnect and delete runtime để giải phóng GPU.
Đăng nhập, quota và mount Drive cần bạn thao tác. Demo GPU→UI và benchmark tracking sáu MOV ở cả 2 FPS/native đã được đo; xem docs/CCTV_FULLVIDEO_COLAB.md để biết timing và giới hạn. M8, review độc lập và máy thứ hai vẫn chưa nghiệm thu.
